# 03 — Classical ML Triage Layer
**RoadSentinel AI** — Trains and benchmarks leak-safe pipelines for SVM, Random Forest, Logistic Regression, and XGBoost.
Generates unified ROC curves, Precision-Recall curves, and Confusion Matrices.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd, joblib
from src.preprocessing import split_first
from src.models import train_svm, train_rf, train_logreg, train_xgb, evaluate_and_compare_models
from src.imbalance import compute_scale_pos_weight

## 1. Load Data & Perform Leak-Safe Split
Always split BEFORE fitting preprocessing transformers.

In [ ]:
df = pd.read_csv('../data/engineered_features.csv')
X_train, X_test, y_train, y_test = split_first(df, label_col='is_accident')
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}, Positive rate: {y_train.mean():.2%}")

## 2. Train Classifiers Inside Leak-Safe Pipelines

In [ ]:
print("Training SVM (with GridSearch)...\n")
svm_pipe = train_svm(X_train, y_train)

print("Training Random Forest...\n")
rf_pipe = train_rf(X_train, y_train)

print("Training Logistic Regression...\n")
logreg_pipe = train_logreg(X_train, y_train)

print("Training XGBoost...\n")
xgb_pipe = train_xgb(X_train, y_train, scale_pos_weight=compute_scale_pos_weight(y_train))

models = {
    'SVM': svm_pipe,
    'RandomForest': rf_pipe,
    'LogisticRegression': logreg_pipe,
    'XGBoost': xgb_pipe,
}

## 3. Comprehensive Evaluation & Benchmark Suite

In [ ]:
metrics_df = evaluate_and_compare_models(models, X_test, y_test, save_dir='../models/metrics')
metrics_df

## 4. Save Production Models

In [ ]:
joblib.dump(svm_pipe, '../models/svm_triage_pipeline.joblib')
joblib.dump(xgb_pipe, '../models/xgb_triage_pipeline.joblib')
print("Models saved to ../models/")